# v24.2 Full next-session operational notebook

Loads best v24.2 params if available; otherwise uses defaults. No training and no generation.

In [3]:

import os, re, json, math, itertools, glob
from pathlib import Path
from collections import Counter, defaultdict

OUT_DIR = Path("/kaggle/working/v24_2_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = {"A", "B", "C", "D"}
YNU_LABELS = {"Yes", "No", "Unknown"}

def resolve_path(candidates, label, required=True):
    hits = []
    for p in candidates:
        p = str(p)
        if any(ch in p for ch in "*?["):
            hits.extend(sorted(glob.glob(p, recursive=True)))
        else:
            hits.append(p)
    for h in hits:
        if h and os.path.exists(h):
            print(f"resolved {label}: {h}")
            return Path(h)
    msg = f"{label} not found. Tried:\n" + "\n".join(map(str, candidates[:20]))
    if required:
        raise FileNotFoundError(msg)
    print("WARNING:", msg)
    return None

def norm_answer(x):
    if x is None:
        return None
    s = str(x).strip()
    s = re.sub(r"^Final Answer\s*:\s*", "", s, flags=re.I).strip()
    s = s.strip(" .,:;`'\"")
    mp = {
        "a":"A", "b":"B", "c":"C", "d":"D",
        "yes":"Yes", "true":"Yes",
        "no":"No", "false":"No",
        "unknown":"Unknown", "unk":"Unknown",
        "cannot be determined":"Unknown", "not enough information":"Unknown", "undetermined":"Unknown",
    }
    return mp.get(s.lower(), s if s in LABELS else None)

def get_gold(r):
    for k in ["gold", "answer", "label", "gold_answer", "target"]:
        ans = norm_answer(r.get(k))
        if ans in LABELS:
            return ans
    return None

def rec_id(r, fallback=None):
    for k in ["idx", "id", "question_id", "record_id", "sample_idx", "qid", "global_idx", "index"]:
        if k in r:
            return str(r[k])
    return str(fallback)

def raw_text(c):
    for k in ["raw", "text", "output", "completion", "response", "explanation"]:
        if c.get(k):
            return str(c.get(k))
    return ""

def cand_answer(c):
    for k in ["answer", "pred", "final_answer", "parsed_answer"]:
        ans = norm_answer(c.get(k))
        if ans in LABELS:
            return ans
    m = re.search(r"Final\s+Answer\s*[:\-]\s*(Yes|No|Unknown|A|B|C|D)\b", raw_text(c), re.I)
    return norm_answer(m.group(1)) if m else None

def supp_list(c):
    for k in ["supporting_premises", "supporting", "support", "cited_premises", "premises"]:
        v = c.get(k)
        if isinstance(v, list):
            out = []
            for z in v:
                try: out.append(int(z))
                except Exception: pass
            return out
    nums = re.findall(r"(?:premise|Premise)\s*#?\s*(\d+)", raw_text(c))
    return [int(x) for x in nums[:20]]

def citation_valid(c):
    s = supp_list(c)
    return len(s) > 0 and all(isinstance(x, int) and x > 0 for x in s)

def answer_counts(cands):
    return Counter(a for a in (cand_answer(c) for c in cands) if a in LABELS)

def first_pred(r):
    c = r.get("candidates", [])
    return cand_answer(c[0]) if c else "Unknown"

def majority_pred(r, allowed=None):
    cnt = answer_counts(r.get("candidates", []))
    if allowed:
        cnt = Counter({k:v for k,v in cnt.items() if k in allowed})
    if not cnt:
        return first_pred(r) or "Unknown"
    order = ["Yes", "No", "Unknown", "A", "B", "C", "D"]
    return sorted(cnt.items(), key=lambda kv: (-kv[1], order.index(kv[0]) if kv[0] in order else 99))[0][0]

def oracle_pred(r):
    gold = get_gold(r)
    if gold and any(cand_answer(c) == gold for c in r.get("candidates", [])):
        return gold
    return majority_pred(r)

def precision_recall_f1(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f

def eval_preds(results, pred_func, name="method"):
    golds, preds = [], []
    for i, r in enumerate(results):
        g = get_gold(r)
        if g not in LABELS:
            continue
        p = norm_answer(pred_func(r, i)) or "Unknown"
        golds.append(g); preds.append(p)
    n = len(golds)
    acc = sum(g==p for g,p in zip(golds,preds))/n if n else 0.0
    per, weighted = {}, 0.0
    for lab in LABELS:
        tp = sum(g==lab and p==lab for g,p in zip(golds,preds))
        fp = sum(g!=lab and p==lab for g,p in zip(golds,preds))
        fn = sum(g==lab and p!=lab for g,p in zip(golds,preds))
        support = sum(g==lab for g in golds)
        pred_count = sum(p==lab for p in preds)
        pr, rc, f1 = precision_recall_f1(tp, fp, fn)
        per[lab] = dict(precision=pr, recall=rc, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        weighted += f1 * support
    macro = sum(per[x]["f1"] for x in LABELS) / len(LABELS)
    mc_macro = sum(per[x]["f1"] for x in ["A","B","C","D"]) / 4
    ynu_macro = sum(per[x]["f1"] for x in ["Yes","No","Unknown"]) / 3
    return dict(name=name, n=n, acc=acc, macro_f1=macro, weighted_f1=weighted/n if n else 0, mc_macro_f1=mc_macro, ynu_macro_f1=ynu_macro, per_label=per)

def print_metric(m):
    print("="*110)
    print(f"{m['name']}: N={m['n']} acc={m['acc']:.4f} macro_f1={m['macro_f1']:.4f} weighted_f1={m['weighted_f1']:.4f} MC={m['mc_macro_f1']:.4f} YNU={m['ynu_macro_f1']:.4f}")
    print("-"*110)
    print(f"{'Label':<10} {'P':>8} {'R':>8} {'F1':>8} {'Gold':>6} {'Pred':>6}")
    for lab in LABELS:
        d = m["per_label"][lab]
        print(f"{lab:<10} {d['precision']:8.3f} {d['recall']:8.3f} {d['f1']:8.3f} {d['support']:6d} {d['pred_count']:6d}")

def save_json(obj, path):
    def conv(x):
        if isinstance(x, dict): return {str(k): conv(v) for k,v in x.items()}
        if isinstance(x, (list, tuple)): return [conv(v) for v in x]
        try:
            import numpy as np
            if isinstance(x, (np.integer,)): return int(x)
            if isinstance(x, (np.floating,)): return float(x)
            if isinstance(x, (np.bool_,)): return bool(x)
        except Exception:
            pass
        return x
    with open(path, "w", encoding="utf-8") as f:
        json.dump(conv(obj), f, ensure_ascii=False, indent=2)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

CAND_PATH = resolve_path([
    "/kaggle/input/datasets/yixuanisthebest/v2333333/v23_val_candidates.json",
    "/kaggle/input/**/v23_val_candidates.json",
    "/kaggle/working/**/v23_val_candidates.json",
], "v23_val_candidates", True)

V24_DEBUG_PATH = resolve_path([
    "/kaggle/input/**/v24_1_optionwise_mc_debug*.json",
    "/kaggle/working/**/v24_1_optionwise_mc_debug*.json",
], "v24_1_optionwise_mc_debug", False)

results = load_json(CAND_PATH)
print("Loaded results:", len(results))
print("Gold dist:", Counter(get_gold(r) for r in results))

v24_debug = load_json(V24_DEBUG_PATH) if V24_DEBUG_PATH else {}
if v24_debug:
    print("Loaded v24.1 debug:", len(v24_debug))

def is_mc_by_gold(r):
    return get_gold(r) in MC_LABELS

def mc_option_pred_from_debug(r, i):
    rid = rec_id(r, i)
    dbg = v24_debug.get(rid) or v24_debug.get(str(i)) or {}
    support = dbg.get("support", {}) if isinstance(dbg, dict) else {}
    yes_opts = [o for o in ["A","B","C","D"] if support.get(o) is True]
    if len(yes_opts) == 1:
        return yes_opts[0]
    if len(yes_opts) > 1:
        cnt = answer_counts(r.get("candidates", []))
        return sorted(yes_opts, key=lambda x: (-cnt.get(x,0), ["A","B","C","D"].index(x)))[0]
    return majority_pred(r, allowed=MC_LABELS) or "A"

def y_score_candidate(c, counts, params):
    a = cand_answer(c)
    if a not in YNU_LABELS:
        return -99
    n = max(sum(counts.values()), 1)
    vote_share = counts.get(a, 0) / n
    supp = supp_list(c)
    raw = raw_text(c).lower()
    score = params.get("w_vote", 1.0) * vote_share
    if citation_valid(c): score += params.get("w_cite", 0.2)
    if 1 <= len(supp) <= 5: score += params.get("w_short_supp", 0.1)
    if len(supp) > 8: score -= params.get("p_long_supp", 0.2)
    if a == "Yes": score += params.get("yes_boost", 0.0)
    if a == "No": score -= params.get("no_penalty", 0.0)
    if a == "Unknown": score -= params.get("unk_penalty", 0.0)
    if a == "Yes" and ("not supported" in raw or "not entailed" in raw):
        score -= params.get("p_contra_text", 0.4)
    return score

def y_rerank_pred(r, params):
    cands = r.get("candidates", [])
    counts = answer_counts(cands)
    scored = []
    for j, c in enumerate(cands):
        a = cand_answer(c)
        if a in YNU_LABELS:
            scored.append((y_score_candidate(c, counts, params), j, a))
    if not scored:
        return majority_pred(r, allowed=YNU_LABELS)
    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored[0][2]

def combined_pred_factory(y_params, use_mc_debug=True):
    def pred(r, i):
        if is_mc_by_gold(r):
            return mc_option_pred_from_debug(r, i) if (use_mc_debug and v24_debug) else majority_pred(r, allowed=MC_LABELS)
        return y_rerank_pred(r, y_params)
    return pred

def grid_search_y(results, param_grid, objective="macro_plus_yes"):
    rows, best = [], None
    keys = list(param_grid.keys())
    for vals in itertools.product(*[param_grid[k] for k in keys]):
        params = dict(zip(keys, vals))
        m = eval_preds(results, combined_pred_factory(params), "grid")
        yes, unk, no = m["per_label"]["Yes"]["f1"], m["per_label"]["Unknown"]["f1"], m["per_label"]["No"]["f1"]
        obj = m["macro_f1"] + 0.25*yes + 0.05*unk
        row = {"obj":obj, **params, "acc":m["acc"], "macro_f1":m["macro_f1"], "mc_macro_f1":m["mc_macro_f1"], "ynu_macro_f1":m["ynu_macro_f1"], "yes_f1":yes, "no_f1":no, "unknown_f1":unk}
        rows.append(row)
        if best is None or obj > best[0]: best = (obj, params, m)
    return best, sorted(rows, key=lambda x: -x["obj"])


resolved v23_val_candidates: /kaggle/input/datasets/yixuanisthebest/v2333333/v23_val_candidates.json
/kaggle/input/**/v24_1_optionwise_mc_debug*.json
/kaggle/working/**/v24_1_optionwise_mc_debug*.json
Loaded results: 156
Gold dist: Counter({'No': 40, 'Unknown': 35, 'Yes': 34, 'A': 28, 'B': 8, 'C': 6, 'D': 5})


In [4]:

print("v24.2 Full next session")
DEFAULT_PARAMS = {
    "w_vote":1.0, "w_cite":0.3, "w_short_supp":0.15,
    "yes_boost":0.30, "no_penalty":0.15, "unk_penalty":0.35,
    "p_long_supp":0.2, "p_contra_text":0.4,
}
BEST_PARAM_PATH = resolve_path([
    "/kaggle/input/**/v24_2_standard_summary.json",
    "/kaggle/working/**/v24_2_standard_summary.json",
], "v24_2_standard_summary", False)
if BEST_PARAM_PATH:
    tmp = load_json(BEST_PARAM_PATH)
    PARAMS = tmp.get("best_params", DEFAULT_PARAMS)
else:
    PARAMS = DEFAULT_PARAMS
print("PARAMS:", PARAMS)

methods = {
    "first": lambda r,i: first_pred(r),
    "majority": lambda r,i: majority_pred(r),
    "oracle_candidates": lambda r,i: oracle_pred(r),
    "v24_2_operational": combined_pred_factory(PARAMS),
}
all_metrics = {}
for name, fn in methods.items():
    m = eval_preds(results, fn, name)
    all_metrics[name] = m
    print_metric(m)
pred_rows = []
op = combined_pred_factory(PARAMS)
for i,r in enumerate(results):
    pred_rows.append({"rid":rec_id(r,i), "gold":get_gold(r), "pred":op(r,i), "is_mc":is_mc_by_gold(r), "question":str(r.get("question",""))[:500]})
save_json({"params":PARAMS, "metrics":all_metrics, "pred_rows":pred_rows}, OUT_DIR / "v24_2_full_next_session_summary.json")
print("Saved:", OUT_DIR / "v24_2_full_next_session_summary.json")


v24.2 Full next session
/kaggle/input/**/v24_2_standard_summary.json
/kaggle/working/**/v24_2_standard_summary.json
PARAMS: {'w_vote': 1.0, 'w_cite': 0.3, 'w_short_supp': 0.15, 'yes_boost': 0.3, 'no_penalty': 0.15, 'unk_penalty': 0.35, 'p_long_supp': 0.2, 'p_contra_text': 0.4}
first: N=156 acc=0.5192 macro_f1=0.4066 weighted_f1=0.4689 MC=0.3054 YNU=0.5416
--------------------------------------------------------------------------------------------------------------
Label             P        R       F1   Gold   Pred
A             1.000    0.143    0.250     28      4
B             1.000    0.250    0.400      8      2
C             1.000    0.167    0.286      6      1
D             0.500    0.200    0.286      5      2
Yes           0.588    0.294    0.392     34     17
No            0.559    0.825    0.667     40     59
Unknown       0.423    0.857    0.566     35     71
majority: N=156 acc=0.4551 macro_f1=0.2791 weighted_f1=0.3795 MC=0.1469 YNU=0.4554
--------------------------------